# predicting neurodevelopmental and MH conditions frorm c4

In [ ]:
# Imports for co-occurring conditions prediction
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("=== CO-OCCURRING CONDITIONS PREDICTION EXPERIMENT ===")

€ 2. load data and initial exploration 

In [ ]:
# Load the original C4 dataset with diagnosis information
df_raw = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/raw/data_c4_raw.csv')

print("=== DATASET EXPLORATION ===")
print(f"Original dataset shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")

# Remove test entries (first 16 rows)
df_raw = df_raw.iloc[16:].reset_index(drop=True)

# Remove records with missing compulsory data (opted out of data sharing)
compulsory_cols = ['age', 'sex', 'handedness', 'education', 'occupation', 'country_region']
df_clean = df_raw.dropna(subset=compulsory_cols)

print(f"After removing test entries and opt-outs: {df_clean.shape}")

# Explore diagnosis columns
diagnosis_cols = [col for col in df_clean.columns if 'diagnosis' in col]
print(f"Diagnosis columns: {diagnosis_cols}")

# Show diagnosis distribution
for col in diagnosis_cols:
    if col in df_clean.columns:
        print(f"{col} distribution:")
        print(df_clean[col].value_counts().head(10))

# 3. create autism only dataset

In [ ]:
# Create autism-only dataset
print("=== CREATING AUTISM-ONLY DATASET ===")

# Filter to autistic population only
autism_mask = (df_clean['diagnosis_0'] == 2) | (df_clean['diagnosis_1'] == 2) | (df_clean['diagnosis_2'] == 2) | \
              (df_clean['diagnosis_3'] == 2) | (df_clean['diagnosis_4'] == 2) | (df_clean['diagnosis_5'] == 2) | \
              (df_clean['diagnosis_6'] == 2) | (df_clean['diagnosis_7'] == 2) | (df_clean['diagnosis_8'] == 2) | \
              (df_clean['autism_diagnosis_0'] == 1) | (df_clean['autism_diagnosis_0'] == 2) | (df_clean['autism_diagnosis_0'] == 3) | \
              (df_clean['autism_diagnosis_1'] == 1) | (df_clean['autism_diagnosis_1'] == 2) | (df_clean['autism_diagnosis_1'] == 3) | \
              (df_clean['autism_diagnosis_2'] == 1) | (df_clean['autism_diagnosis_2'] == 2) | (df_clean['autism_diagnosis_2'] == 3)

df_autism_only = df_clean[autism_mask].copy()
print(f"Autistic population: {len(df_autism_only):,} cases")

# Show demographics of autistic population
print(f"Autistic population demographics:")
print(f"Age: {df_autism_only['age'].mean():.1f} ± {df_autism_only['age'].std():.1f}")
print(f"Sex distribution:")
print(df_autism_only['sex'].value_counts())
print(f"Education distribution:")
print(df_autism_only['education'].value_counts())

# 4. co-occuring conditions targets

In [ ]:
# Create targets for ALL co-occurring conditions in autistic population
print("=== CREATING ALL CO-OCCURRING CONDITION TARGETS ===")

autism_targets = {}

# ADHD in autistic population (code 1)
adhd_autism_mask = (df_autism_only['diagnosis_0'] == 1) | (df_autism_only['diagnosis_1'] == 1) | \
                   (df_autism_only['diagnosis_2'] == 1) | (df_autism_only['diagnosis_3'] == 1) | \
                   (df_autism_only['diagnosis_4'] == 1) | (df_autism_only['diagnosis_5'] == 1) | \
                   (df_autism_only['diagnosis_6'] == 1) | (df_autism_only['diagnosis_7'] == 1) | \
                   (df_autism_only['diagnosis_8'] == 1)
autism_targets['ADHD'] = adhd_autism_mask.astype(int)

# Bipolar in autistic population (code 3)
bipolar_autism_mask = (df_autism_only['diagnosis_0'] == 3) | (df_autism_only['diagnosis_1'] == 3) | \
                     (df_autism_only['diagnosis_2'] == 3) | (df_autism_only['diagnosis_3'] == 3) | \
                     (df_autism_only['diagnosis_4'] == 3) | (df_autism_only['diagnosis_5'] == 3) | \
                     (df_autism_only['diagnosis_6'] == 3) | (df_autism_only['diagnosis_7'] == 3) | \
                     (df_autism_only['diagnosis_8'] == 3)
autism_targets['Bipolar'] = bipolar_autism_mask.astype(int)

# Depression in autistic population (code 4)
depression_autism_mask = (df_autism_only['diagnosis_0'] == 4) | (df_autism_only['diagnosis_1'] == 4) | \
                        (df_autism_only['diagnosis_2'] == 4) | (df_autism_only['diagnosis_3'] == 4) | \
                        (df_autism_only['diagnosis_4'] == 4) | (df_autism_only['diagnosis_5'] == 4) | \
                        (df_autism_only['diagnosis_6'] == 4) | (df_autism_only['diagnosis_7'] == 4) | \
                        (df_autism_only['diagnosis_8'] == 4)
autism_targets['Depression'] = depression_autism_mask.astype(int)

# Learning disability in autistic population (code 5)
learning_autism_mask = (df_autism_only['diagnosis_0'] == 5) | (df_autism_only['diagnosis_1'] == 5) | \
                      (df_autism_only['diagnosis_2'] == 5) | (df_autism_only['diagnosis_3'] == 5) | \
                      (df_autism_only['diagnosis_4'] == 5) | (df_autism_only['diagnosis_5'] == 5) | \
                      (df_autism_only['diagnosis_6'] == 5) | (df_autism_only['diagnosis_7'] == 5) | \
                      (df_autism_only['diagnosis_8'] == 5)
autism_targets['Learning'] = learning_autism_mask.astype(int)

# OCD in autistic population (code 6)
ocd_autism_mask = (df_autism_only['diagnosis_0'] == 6) | (df_autism_only['diagnosis_1'] == 6) | \
                  (df_autism_only['diagnosis_2'] == 6) | (df_autism_only['diagnosis_3'] == 6) | \
                  (df_autism_only['diagnosis_4'] == 6) | (df_autism_only['diagnosis_5'] == 6) | \
                  (df_autism_only['diagnosis_6'] == 6) | (df_autism_only['diagnosis_7'] == 6) | \
                  (df_autism_only['diagnosis_8'] == 6)
autism_targets['OCD'] = ocd_autism_mask.astype(int)

# Schizophrenia in autistic population (code 7)
schizophrenia_autism_mask = (df_autism_only['diagnosis_0'] == 7) | (df_autism_only['diagnosis_1'] == 7) | \
                           (df_autism_only['diagnosis_2'] == 7) | (df_autism_only['diagnosis_3'] == 7) | \
                           (df_autism_only['diagnosis_4'] == 7) | (df_autism_only['diagnosis_5'] == 7) | \
                           (df_autism_only['diagnosis_6'] == 7) | (df_autism_only['diagnosis_7'] == 7) | \
                           (df_autism_only['diagnosis_8'] == 7)
autism_targets['Schizophrenia'] = schizophrenia_autism_mask.astype(int)

print("=== ALL CO-OCCURRING CONDITIONS IN AUTISTIC POPULATION ===")
for condition, target in autism_targets.items():
    prevalence = target.mean()
    print(f"{condition}: {target.sum():,} cases ({prevalence:.3f} prevalence)")
    df_autism_only[f'{condition}_target'] = target

# Show which conditions have sufficient samples for modeling
print("=== MODELING RECOMMENDATIONS ===")
for condition, target in autism_targets.items():
    if target.sum() >= 1000:
        print(f"SUITABLE: {condition}: {target.sum():,} cases - SUITABLE FOR MODELING")
    else:
        print(f"INSUFFICIENT: {condition}: {target.sum():,} cases - INSUFFICIENT SAMPLES")

# Create target for ANY co-occurring condition
print("\n=== CREATING ANY CO-OCCURRING CONDITION TARGET ===")

# Create the any_condition target by combining all individual conditions
any_condition_mask = (df_autism_only['ADHD_target'] | 
                     df_autism_only['Bipolar_target'] | 
                     df_autism_only['Depression_target'] | 
                     df_autism_only['Learning_target'] | 
                     df_autism_only['OCD_target'] | 
                     df_autism_only['Schizophrenia_target'])

df_autism_only['Any_condition_target'] = any_condition_mask.astype(int)

# Analyze the any_condition target
any_condition_prevalence = any_condition_mask.mean()
any_condition_count = any_condition_mask.sum()

print(f"ANY CO-OCCURRING CONDITION ANALYSIS:")
print(f"Total cases with any co-occurring condition: {any_condition_count:,}")
print(f"Prevalence: {any_condition_prevalence:.3f} ({any_condition_prevalence*100:.1f}%)")

# Show breakdown by individual conditions
print(f"\nBREAKDOWN BY INDIVIDUAL CONDITIONS:")
for condition in ['ADHD_target', 'Bipolar_target', 'Depression_target', 'Learning_target', 'OCD_target', 'Schizophrenia_target']:
    condition_count = df_autism_only[condition].sum()
    condition_prevalence = df_autism_only[condition].mean()
    print(f"  {condition}: {condition_count:,} cases ({condition_prevalence:.3f} prevalence)")

# Show overlap analysis - FIXED VERSION
print(f"\nOVERLAP ANALYSIS:")
# Create a DataFrame with all condition targets for overlap analysis
condition_targets_df = df_autism_only[['ADHD_target', 'Bipolar_target', 'Depression_target', 
                                      'Learning_target', 'OCD_target', 'Schizophrenia_target']]

# Count how many conditions each person has
condition_counts = condition_targets_df.sum(axis=1)

print(f"Cases with exactly 1 condition: {(condition_counts == 1).sum():,}")
print(f"Cases with exactly 2 conditions: {(condition_counts == 2).sum():,}")
print(f"Cases with exactly 3 conditions: {(condition_counts == 3).sum():,}")
print(f"Cases with 4+ conditions: {(condition_counts >= 4).sum():,}")

# Add to modeling recommendations
print(f"\n=== UPDATED MODELING RECOMMENDATIONS ===")
if any_condition_count >= 1000:
    print(f"SUITABLE: Any_condition_target: {any_condition_count:,} cases - SUITABLE FOR MODELING")
else:
    print(f"INSUFFICIENT: Any_condition_target: {any_condition_count:,} cases - INSUFFICIENT SAMPLES")

# 5. feature engineering 

In [ ]:
# Feature Engineering - CLINICALLY GROUNDED FOR AUTISM RESEARCH (NO LEAKAGE)
print("=== CLINICALLY-GROUNDED FEATURE ENGINEERING ===")

# First, calculate total scores if they don't exist
if 'aq_total' not in df_autism_only.columns:
    df_autism_only['aq_total'] = df_autism_only[['aq_1', 'aq_2', 'aq_3', 'aq_4', 'aq_5', 'aq_6', 'aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
    print("Calculated AQ total")

if 'eq_total' not in df_autism_only.columns:
    df_autism_only['eq_total'] = df_autism_only[['eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5', 'eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10']].sum(axis=1)
    print("Calculated EQ total")

if 'sqr_total' not in df_autism_only.columns:
    df_autism_only['sqr_total'] = df_autism_only[['sqr_1', 'sqr_2', 'sqr_3', 'sqr_4', 'sqr_5', 'sqr_6', 'sqr_7', 'sqr_8', 'sqr_9', 'sqr_10']].sum(axis=1)
    print("Calculated SQR total")

if 'spq_total' not in df_autism_only.columns:
    df_autism_only['spq_total'] = df_autism_only[['spq_1', 'spq_2', 'spq_3', 'spq_4', 'spq_5', 'spq_6', 'spq_7', 'spq_8', 'spq_9', 'spq_10']].sum(axis=1)
    print("Calculated SPQ total")

# Now calculate subdomain scores if they don't exist
if 'aq_social_skills' not in df_autism_only.columns:
    df_autism_only['aq_social_skills'] = df_autism_only[['aq_1', 'aq_2', 'aq_4']].sum(axis=1)
    df_autism_only['aq_attention_switching'] = df_autism_only[['aq_3', 'aq_5', 'aq_6']].sum(axis=1)
    df_autism_only['aq_attention_to_detail'] = df_autism_only[['aq_7', 'aq_8', 'aq_9', 'aq_10']].sum(axis=1)
    print("Calculated AQ subdomain scores")

if 'eq_cognitive' not in df_autism_only.columns:
    df_autism_only['eq_cognitive'] = df_autism_only[['eq_1', 'eq_2', 'eq_3', 'eq_4', 'eq_5']].sum(axis=1)
    df_autism_only['eq_affective'] = df_autism_only[['eq_6', 'eq_7', 'eq_8', 'eq_9', 'eq_10']].sum(axis=1)
    print("Calculated EQ subdomain scores")

if 'sqr_social_awareness' not in df_autism_only.columns:
    df_autism_only['sqr_social_awareness'] = df_autism_only[['sqr_1', 'sqr_2']].sum(axis=1)
    df_autism_only['sqr_social_cognition'] = df_autism_only[['sqr_3', 'sqr_4', 'sqr_5']].sum(axis=1)
    df_autism_only['sqr_social_communication'] = df_autism_only[['sqr_6', 'sqr_7', 'sqr_8']].sum(axis=1)
    df_autism_only['sqr_social_motivation'] = df_autism_only[['sqr_9', 'sqr_10']].sum(axis=1)
    print("Calculated SQR subdomain scores")

if 'spq_cognitive_perceptual' not in df_autism_only.columns:
    df_autism_only['spq_cognitive_perceptual'] = df_autism_only[['spq_1', 'spq_2', 'spq_3', 'spq_4']].sum(axis=1)
    df_autism_only['spq_interpersonal'] = df_autism_only[['spq_5', 'spq_6', 'spq_7', 'spq_8']].sum(axis=1)
    df_autism_only['spq_disorganized'] = df_autism_only[['spq_9', 'spq_10']].sum(axis=1)
    print("Calculated SPQ subdomain scores")

# 1. AUTISM-SPECIFIC FEATURES (based on autism research)
# Social communication difficulties
df_autism_only['social_communication_deficit'] = df_autism_only['sqr_total'] - df_autism_only['eq_total']
df_autism_only['social_skills_ratio'] = df_autism_only['aq_social_skills'] / (df_autism_only['eq_total'] + 1e-8)

# Sensory processing (SPQ captures this)
df_autism_only['sensory_processing_score'] = df_autism_only['spq_cognitive_perceptual']
df_autism_only['sensory_overload_risk'] = (df_autism_only['spq_cognitive_perceptual'] > 
                                          df_autism_only['spq_cognitive_perceptual'].quantile(0.75)).astype(int)

# Executive function difficulties
df_autism_only['executive_function_deficit'] = df_autism_only['aq_attention_switching'] + df_autism_only['aq_attention_to_detail']
df_autism_only['cognitive_rigidity'] = df_autism_only['aq_attention_to_detail'] / (df_autism_only['aq_attention_switching'] + 1e-8)

# 2. MENTAL HEALTH RISK FACTORS (based on clinical literature)
# Age-related risk factors
df_autism_only['adolescent_risk'] = ((df_autism_only['age'] >= 12) & (df_autism_only['age'] <= 18)).astype(int)
df_autism_only['young_adult_risk'] = ((df_autism_only['age'] >= 18) & (df_autism_only['age'] <= 25)).astype(int)
df_autism_only['adult_risk'] = (df_autism_only['age'] > 25).astype(int)

# Gender-related risk factors
df_autism_only['female_autism'] = (df_autism_only['sex'] == 2).astype(int)
df_autism_only['male_autism'] = (df_autism_only['sex'] == 1).astype(int)

# Education and employment risk factors
df_autism_only['education_risk'] = (df_autism_only['education'] <= 2).astype(int)
df_autism_only['stem_occupation'] = (df_autism_only['occupation'] == 3).astype(int)

# 3. CLINICAL SUBTYPE FEATURES
# Autism severity indicators
df_autism_only['high_aq_severity'] = (df_autism_only['aq_total'] > df_autism_only['aq_total'].quantile(0.75)).astype(int)
df_autism_only['low_eq_severity'] = (df_autism_only['eq_total'] < df_autism_only['eq_total'].quantile(0.25)).astype(int)
df_autism_only['high_spq_severity'] = (df_autism_only['spq_total'] > df_autism_only['spq_total'].quantile(0.75)).astype(int)

# Autism subtype (based on questionnaire patterns)
df_autism_only['social_autism'] = ((df_autism_only['aq_social_skills'] > df_autism_only['aq_social_skills'].quantile(0.75)) & 
                                   (df_autism_only['eq_total'] < df_autism_only['eq_total'].quantile(0.25))).astype(int)
df_autism_only['cognitive_autism'] = ((df_autism_only['aq_attention_to_detail'] > df_autism_only['aq_attention_to_detail'].quantile(0.75)) & 
                                      (df_autism_only['spq_cognitive_perceptual'] > df_autism_only['spq_cognitive_perceptual'].quantile(0.75))).astype(int)

# 4. ENVIRONMENTAL AND DEMOGRAPHIC RISK FACTORS
# Geographic risk factors
df_autism_only['urban_risk'] = (df_autism_only['country_region'].isin([4, 5, 6, 7, 8, 9, 10, 11])).astype(int)
df_autism_only['rural_risk'] = (df_autism_only['country_region'].isin([1, 2, 3, 12, 13])).astype(int)

# Age-sex interaction risk
df_autism_only['age_sex_risk'] = df_autism_only['age'] * df_autism_only['sex']

# 5. QUESTIONNAIRE-BASED RISK SCORES
# Social anxiety risk
df_autism_only['social_anxiety_risk'] = (df_autism_only['sqr_social_awareness'] + df_autism_only['sqr_social_cognition']) / 2

# Depression risk indicators
df_autism_only['depression_risk_score'] = (df_autism_only['eq_total'] - df_autism_only['sqr_total']) / (df_autism_only['eq_total'] + 1e-8)

# ADHD risk indicators
df_autism_only['adhd_risk_score'] = df_autism_only['aq_attention_switching'] / (df_autism_only['aq_total'] + 1e-8)

print(f"After clinically-grounded feature engineering: {df_autism_only.shape}")
print(f"New features created: {df_autism_only.shape[1] - 114}")

# 6. clean data preperation

In [ ]:
# CLEAN data preparation function - NO LEAKAGE
def prepare_data_for_condition_clean(df, target_col, min_samples=1000):
    """Prepare data for a specific condition prediction - NO LEAKAGE"""
    
    # Check if we have enough samples
    if target_col not in df.columns:
        return None, None, None, None, None
    
    target_counts = df[target_col].value_counts()
    if target_counts.min() < min_samples:
        print(f"WARNING: {target_col}: Insufficient samples ({target_counts.min()} < {min_samples})")
        return None, None, None, None, None
    
    # Prepare features and target
    y = df[target_col]
    
    # CRITICAL: Remove ALL target and diagnosis columns
    exclude_cols = []
    for col in df.columns:
        if 'target' in col or 'diagnosis' in col or 'autism_diagnosis' in col:
            exclude_cols.append(col)
    
    X = df.drop(exclude_cols, axis=1)
    
    print(f"Removed {len(exclude_cols)} potential leakage columns: {exclude_cols[:5]}...")
    
    # Remove non-numeric and constant features
    X = X.select_dtypes(include=[np.number])
    constant_features = X.columns[X.std() == 0]
    X = X.drop(columns=constant_features)
    
    # Handle missing values
    X = X.fillna(X.mean())
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"SUCCESS: {target_col}: {len(X_train)} train, {len(X_test)} test samples")
    print(f"   Prevalence: {y_train.mean():.3f} train, {y_test.mean():.3f} test")
    print(f"   Features used: {len(X.columns)}")
    
    return X_train_scaled, X_test_scaled, y_train, y_test, X.columns

# Model training function
def train_and_evaluate_models_clean(X_train, X_test, y_train, y_test, condition_name):
    """Train and evaluate multiple models for a condition - NO LEAKAGE"""
    
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
        'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'),
        'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1, scale_pos_weight=10),
        'LightGBM': lgb.LGBMClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1, class_weight='balanced')
    }
    
    results = []
    
    print(f"TRAINING MODELS FOR {condition_name.upper()}")
    
    for name, model in models.items():
        print(f"  Training {name}...")
        
        # Train model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        y_probs = model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_probs)
        f1 = f1_score(y_test, y_pred)
        
        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring='f1', n_jobs=-1)
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        results.append({
            'Condition': condition_name,
            'Model': name,
            'Accuracy': accuracy,
            'AUC': auc,
            'F1': f1,
            'CV_F1_Mean': cv_mean,
            'CV_F1_Std': cv_std
        })
        
        print(f"    {name}: F1={f1:.4f}, AUC={auc:.4f}, CV_F1={cv_mean:.4f} (±{cv_std:.4f})")
    
    return pd.DataFrame(results)

# 7. model training and eval

In [ ]:
# Optimized model training for autism-focused prediction - NO LEAKAGE
print("=== AUTISM-FOCUSED CO-OCCURRING CONDITIONS PREDICTION (NO LEAKAGE) ===")

# Define ALL conditions to test (including any condition)
conditions_to_test = ['ADHD_target', 'Bipolar_target', 'Depression_target', 'Learning_target', 'OCD_target', 'Schizophrenia_target', 'Any_condition_target']

autism_results = []

for condition in conditions_to_test:
    print(f"PREDICTING: {condition} IN AUTISTIC POPULATION")
    print("="*60)
    
    # Prepare data for this condition using CLEAN function
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Train and evaluate models
        condition_results = train_and_evaluate_models_clean(X_train, X_test, y_train, y_test, condition)
        
        # Find best model
        best_model_name = condition_results.loc[condition_results['F1'].idxmax(), 'Model']
        best_f1 = condition_results['F1'].max()
        print(f"  Best model: {best_model_name} (F1={best_f1:.4f})")
        
        autism_results.append(condition_results)
        
        # Save individual results
        condition_results.to_csv(f'/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_{condition}_results.csv', index=False)
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Combine all results
if autism_results:
    combined_autism_results = pd.concat(autism_results, ignore_index=True)
    
    print("AUTISM-FOCUSED PREDICTION RESULTS")
    print("="*60)
    
    # Performance by condition
    print("PERFORMANCE BY CONDITION:")
    condition_summary = combined_autism_results.groupby('Condition')[['F1', 'AUC']].mean().sort_values('F1', ascending=False)
    print(condition_summary)
    
    # Save combined results
    combined_autism_results.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_all_conditions_results.csv', index=False)
    print("All autism-focused results saved!")
    
else:
    print("No autism-focused experiments completed successfully")

# 8. feature importance analysis

In [ ]:
# Feature importance analysis
print("=== FEATURE IMPORTANCE ANALYSIS ===")

# Load the best performing model for each condition and analyze feature importance
conditions_to_test = ['ADHD_target', 'Bipolar_target', 'Depression_target', 'Learning_target', 'OCD_target', 'Schizophrenia_target', 'Any_condition_target']

feature_importance_results = {}

for condition in conditions_to_test:
    print(f"ANALYZING FEATURE IMPORTANCE FOR: {condition}")
    print("-" * 50)
    
    # Prepare data
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Train Random Forest for feature importance (most interpretable)
        rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
        rf_model.fit(X_train, y_train)
        
        # Get feature importance
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Show top 20 features
        print(f"Top 20 features for {condition}:")
        print(importance_df.head(20))
        
        # Save feature importance
        importance_df.to_csv(f'/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_{condition}_feature_importance.csv', index=False)
        
        feature_importance_results[condition] = importance_df
        
        # Analyze feature categories
        print(f"Feature category analysis for {condition}:")
        
        # Questionnaire features
        questionnaire_features = [f for f in importance_df['feature'] if any(x in f for x in ['aq_', 'eq_', 'sqr_', 'spq_'])]
        questionnaire_importance = importance_df[importance_df['feature'].isin(questionnaire_features)]['importance'].sum()
        print(f"  Questionnaire features: {len(questionnaire_features)} features, {questionnaire_importance:.3f} total importance")
        
        # Demographic features
        demographic_features = [f for f in importance_df['feature'] if any(x in f for x in ['age', 'sex', 'education', 'occupation', 'country'])]
        demographic_importance = importance_df[importance_df['feature'].isin(demographic_features)]['importance'].sum()
        print(f"  Demographic features: {len(demographic_features)} features, {demographic_importance:.3f} total importance")
        
        # Clinical features
        clinical_features = [f for f in importance_df['feature'] if any(x in f for x in ['severity', 'risk', 'deficit', 'autism'])]
        clinical_importance = importance_df[importance_df['feature'].isin(clinical_features)]['importance'].sum()
        print(f"  Clinical features: {len(clinical_features)} features, {clinical_importance:.3f} total importance")
        
        print()
        
    else:
        print(f"Skipping {condition} - insufficient samples")

# Overall feature importance summary
print("=== OVERALL FEATURE IMPORTANCE SUMMARY ===")

# Combine all feature importances
all_features = set()
for condition, importance_df in feature_importance_results.items():
    all_features.update(importance_df['feature'])

# Calculate average importance across all conditions
overall_importance = pd.DataFrame({'feature': list(all_features)})
overall_importance['avg_importance'] = 0.0
overall_importance['count_conditions'] = 0

for condition, importance_df in feature_importance_results.items():
    for feature in all_features:
        if feature in importance_df['feature'].values:
            importance = importance_df[importance_df['feature'] == feature]['importance'].iloc[0]
            overall_importance.loc[overall_importance['feature'] == feature, 'avg_importance'] += importance
            overall_importance.loc[overall_importance['feature'] == feature, 'count_conditions'] += 1

# Calculate average
overall_importance['avg_importance'] = overall_importance['avg_importance'] / overall_importance['count_conditions']
overall_importance = overall_importance.sort_values('avg_importance', ascending=False)

print("Top 20 most important features across all conditions:")
print(overall_importance.head(20))

# Save overall feature importance
overall_importance.to_csv('/Users/eb2007/playground/bullpy/c4_experiments/data/processed/autism_overall_feature_importance.csv', index=False)

print("Feature importance analysis complete!")

# hyperparam tuning 

In [ ]:
# HYPERPARAMETER TUNING FOR BEST MODELS
print("=== HYPERPARAMETER TUNING ===")

from sklearn.model_selection import RandomizedSearchCV
import time

# Focus on the best performing models from previous results
# Any_condition_target: LightGBM (F1=0.647)
# Depression_target: LightGBM (F1=0.550)
# Schizophrenia_target: Random Forest (F1=0.552)

# Prepare data for the best conditions
best_conditions = ['Any_condition_target', 'Depression_target', 'Schizophrenia_target']
tuned_models = {}

for condition in best_conditions:
    print(f"\nTUNING {condition}")
    print("="*50)
    
    # Prepare data
    data_result = prepare_data_for_condition_clean(df_autism_only, condition, min_samples=500)
    
    if data_result[0] is not None:
        X_train, X_test, y_train, y_test, feature_names = data_result
        
        # Choose model based on previous best performance
        if condition == 'Any_condition_target':
            base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1)
            param_grid = {
                'n_estimators': [100, 200, 300, 500],
                'max_depth': [6, 8, 10, 12],
                'learning_rate': [0.01, 0.05, 0.1, 0.2],
                'num_leaves': [31, 63, 127, 255],
                'min_child_samples': [20, 50, 100],
                'subsample': [0.8, 0.9, 1.0],
                'colsample_bytree': [0.8, 0.9, 1.0]
            }
        elif condition == 'Depression_target':
            base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1)
            param_grid = {
                'n_estimators': [100, 200, 300],
                'max_depth': [6, 8, 10],
                'learning_rate': [0.05, 0.1, 0.2],
                'num_leaves': [31, 63, 127],
                'min_child_samples': [20, 50]
            }
        else:  # Schizophrenia_target
            base_model = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced')
            param_grid = {
                'n_estimators': [100, 200, 300, 500],
                'max_depth': [10, 15, 20, None],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf': [1, 2, 4],
                'max_features': ['sqrt', 'log2', None]
            }
        
        # Perform hyperparameter tuning
        print(f"Tuning {condition}...")
        start_time = time.time()
        
        random_search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_grid,
            n_iter=20,  # Number of parameter settings sampled
            cv=3,
            scoring='f1',
            n_jobs=-1,
            random_state=42,
            verbose=1
        )
        
        random_search.fit(X_train, y_train)
        
        # Evaluate tuned model
        best_model = random_search.best_estimator_
        y_pred = best_model.predict(X_test)
        y_probs = best_model.predict_proba(X_test)[:, 1]
        
        f1_tuned = f1_score(y_test, y_pred)
        auc_tuned = roc_auc_score(y_test, y_probs)
        
        print(f"Best parameters: {random_search.best_params_}")
        print(f"Tuned performance - F1: {f1_tuned:.4f}, AUC: {auc_tuned:.4f}")
        print(f"Tuning time: {time.time() - start_time:.2f} seconds")
        
        # Store tuned model
        tuned_models[condition] = best_model
        
    else:
        print(f"Skipping {condition} - insufficient samples")

print("\n=== HYPERPARAMETER TUNING COMPLETE ===")
print(f"Tuned models saved for: {list(tuned_models.keys())}")

# ensemble methods